# Fairness Analysis

We take the SBERT scores from the previous notebook and check if the score changes when only one demographic signal in the resume is modified. The signals we change are name, pronouns, and university.

In [ ]:
import os
import pandas as pd
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
scores = pd.read_csv("sbert_scores.csv")
print("Scores shape:", scores.shape)
display(scores.head())

In [ ]:
original = (
    scores[scores["version"] == "original"]
    [["resume_id", "job_id", "job_title", "similarity_score"]]
    .rename(columns={"similarity_score": "original_score"})
)
changed = scores[scores["version"] != "original"].rename(
    columns={"similarity_score": "changed_score"}
)

comparison = changed.merge(original, on=["resume_id", "job_id", "job_title"], how="left")
comparison["score_difference"] = comparison["changed_score"] - comparison["original_score"]
comparison["absolute_difference"] = comparison["score_difference"].abs()
display(comparison.head())
print("Comparison rows:", len(comparison))

In [ ]:
fairness_summary = comparison.groupby("changed_signal").agg(
    average_score_difference=("score_difference", "mean"),
    average_absolute_difference=("absolute_difference", "mean"),
    max_absolute_difference=("absolute_difference", "max"),
    min_score_difference=("score_difference", "min"),
    max_score_difference=("score_difference", "max"),
).reset_index()
display(fairness_summary)

## What this tells us

Changing just one demographic signal in the resume does move the SBERT score a little, even though the qualifications stay the same. In our runs the biggest average shift came from changing the name, then the university, then the pronouns.

Our dataset is small, so these are observations from an audit, not strong claims about real hiring systems. We run statistical tests in a separate notebook to check whether these differences are likely to be real signal or just noise.

In [ ]:
comparison.to_csv("results/fairness_comparison.csv", index=False)
fairness_summary.to_csv("results/fairness_summary.csv", index=False)
print("Saved.")

In [ ]:
from google.colab import files
files.download("results/fairness_comparison.csv")
files.download("results/fairness_summary.csv")